# 04 — Feature Store Setup with Feast

**Objective**: Register the gold-layer engineered features (from NB03) in a [Feast](https://feast.dev/) feature store for reproducible feature serving, point-in-time retrieval, and online/offline consistency.

## What Feast Provides

| Capability | Benefit |
|-----------|--------|
| **Feature catalog** | Central registry of all features with metadata |
| **Point-in-time joins** | Prevents data leakage in historical retrieval |
| **Online serving** | Low-latency feature lookup for real-time inference |
| **Offline store** | Batch retrieval for training and batch scoring |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings("ignore")

# Project paths
PROJECT_ROOT = Path("..").resolve()
FEATURES_DIR = PROJECT_ROOT / "data" / "features"       # Gold layer
FEAST_DIR = PROJECT_ROOT / "data" / "feast"              # Feast data
FEATURE_STORE_DIR = PROJECT_ROOT / "feature_store"       # Feast repo config

FEAST_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:      {PROJECT_ROOT}")
print(f"Gold layer:        {FEATURES_DIR}")
print(f"Feast data:        {FEAST_DIR}")
print(f"Feature store repo: {FEATURE_STORE_DIR}")
print(f"\nGold layer files:")
for f in sorted(FEATURES_DIR.glob("*.parquet")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s} {size_mb:>8.1f} MB")

Project root:      /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform
Gold layer:        /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/data/features
Feast data:        /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/data/feast
Feature store repo: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/feature_store

Gold layer files:
  test_features.parquet                        15.5 MB
  train_features.parquet                       85.5 MB


## 1. Setup — Load Gold Layer Features

In [2]:
# Load gold layer training features produced by NB03
train_features = pd.read_parquet(FEATURES_DIR / "train_features.parquet")

print(f"Train features shape: {train_features.shape[0]:,} rows x {train_features.shape[1]} columns")
print(f"\nTarget distribution:")
print(train_features["TARGET"].value_counts())
print(f"\nFirst 5 columns: {list(train_features.columns[:5])}")
print(f"Last 5 columns:  {list(train_features.columns[-5:])}")
print(f"\nKey column (SK_ID_CURR) unique values: {train_features['SK_ID_CURR'].nunique():,}")

train_features.head(3)

Train features shape: 307,511 rows x 389 columns

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

First 5 columns: ['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR']
Last 5 columns:  ['INST_AMT_INSTALMENT_MEAN_IS_MISSING', 'INST_PAYMENT_RATIO_MEAN_IS_MISSING', 'INST_PAYMENT_RATIO_MIN_IS_MISSING', 'INST_NUM_CONTRACTS_IS_MISSING', 'INST_LATE_PAYMENT_RATE_IS_MISSING']

Key column (SK_ID_CURR) unique values: 307,511


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,INST_PAYMENT_DIFF_SUM_IS_MISSING,INST_DAYS_LATE_MEAN_IS_MISSING,INST_DAYS_LATE_MAX_IS_MISSING,INST_LATE_PAYMENT_COUNT_IS_MISSING,INST_AMT_PAYMENT_MEAN_IS_MISSING,INST_AMT_INSTALMENT_MEAN_IS_MISSING,INST_PAYMENT_RATIO_MEAN_IS_MISSING,INST_PAYMENT_RATIO_MIN_IS_MISSING,INST_NUM_CONTRACTS_IS_MISSING,INST_LATE_PAYMENT_RATE_IS_MISSING
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0,0,0,0,0,0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0,0,0,0,0,0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0,0,0,0,0,0


## 2. Prepare Data for Feast

Feast requires:
- An **event_timestamp** column for point-in-time correctness
- A **created** column to track when the feature row was generated

Since this is a static Kaggle dataset (all features are point-in-time at application), we use a fixed timestamp.

In [3]:
# Add timestamp columns required by Feast
feast_df = train_features.copy()

# Use a fixed date — all features represent application-time snapshots
feast_df["event_timestamp"] = pd.Timestamp("2024-01-01", tz="UTC")
feast_df["created"] = pd.Timestamp("2024-01-01", tz="UTC")

# Ensure SK_ID_CURR is int64 (Feast entity key)
feast_df["SK_ID_CURR"] = feast_df["SK_ID_CURR"].astype("int64")

print(f"Feast-ready DataFrame: {feast_df.shape[0]:,} rows x {feast_df.shape[1]} columns")
print(f"\nAdded columns:")
print(f"  event_timestamp: {feast_df['event_timestamp'].dtype} — {feast_df['event_timestamp'].iloc[0]}")
print(f"  created:         {feast_df['created'].dtype} — {feast_df['created'].iloc[0]}")

# Save to Feast data directory
feast_parquet_path = FEAST_DIR / "train_features.parquet"
feast_df.to_parquet(feast_parquet_path, engine="pyarrow", index=False)

size_mb = feast_parquet_path.stat().st_size / 1e6
print(f"\nSaved Feast-ready features: {feast_parquet_path}")
print(f"  Size: {size_mb:.1f} MB")

Feast-ready DataFrame: 307,511 rows x 391 columns

Added columns:
  event_timestamp: datetime64[s, UTC] — 2024-01-01 00:00:00+00:00
  created:         datetime64[s, UTC] — 2024-01-01 00:00:00+00:00

Saved Feast-ready features: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/data/feast/train_features.parquet
  Size: 85.2 MB


## 3. Feature Store Configuration

The feature store is configured via two files in `feature_store/`:

### `feature_store.yaml` — Feast repo config

In [4]:
# Display the feature store configuration
yaml_path = FEATURE_STORE_DIR / "feature_store.yaml"
print(f"Config file: {yaml_path}")
print(f"{'='*60}")
print(yaml_path.read_text())

Config file: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/feature_store/feature_store.yaml
project: credit_risk
provider: local
registry: ../data/feast/registry.db
online_store:
  type: sqlite
  path: ../data/feast/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 2



### `definitions.py` — Entity & Feature View definitions

In [5]:
# Display the feature definitions
defs_path = FEATURE_STORE_DIR / "definitions.py"
print(f"Definitions file: {defs_path}")
print(f"{'='*60}")
print(defs_path.read_text())

Definitions file: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/feature_store/definitions.py
"""
Feast feature definitions for the Credit Risk ML Platform.

Defines entities, data sources, and feature views for serving
engineered features from the gold layer.
"""

from datetime import timedelta

from feast import Entity, FeatureView, FileSource, Field
from feast.types import Float32, Int32, Int64

# ---------------------------------------------------------------------------
# Entity
# ---------------------------------------------------------------------------
applicant = Entity(
    name="applicant",
    join_keys=["SK_ID_CURR"],
    description="Loan applicant identified by SK_ID_CURR",
)

# ---------------------------------------------------------------------------
# Data source  (Feast-prepared parquet with event_timestamp column)
# ---------------------------------------------------------------------------
train_source = FileSource(
    name="train_featur

## 4. Apply Feature Store

Register the entity, data sources, and feature views with the Feast registry.

In [6]:
import sys
sys.path.insert(0, str(FEATURE_STORE_DIR))

from feast import FeatureStore
from definitions import (
    applicant,
    train_source,
    applicant_features,
    bureau_features,
    credit_history_features,
    missing_indicator_features,
)

# Initialize the feature store (repo_path points to directory with feature_store.yaml)
fs = FeatureStore(repo_path=str(FEATURE_STORE_DIR))

# Apply all definitions to the registry
fs.apply([
    applicant,
    train_source,
    applicant_features,
    bureau_features,
    credit_history_features,
    missing_indicator_features,
])

print("Feature store applied successfully!")
print(f"\nRegistry location: {FEAST_DIR / 'registry.db'}")
print(f"\nRegistered entities:")
for entity in fs.list_entities():
    print(f"  - {entity.name} (join_key={entity.join_key})")

print(f"\nRegistered feature views:")
for fv in fs.list_feature_views():
    n_features = len(fv.features)
    print(f"  - {fv.name:30s} ({n_features:>3} features)")

/home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/feature_store/definitions.py:16: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'applicant'.
  applicant = Entity(


Feature store applied successfully!

Registry location: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/data/feast/registry.db

Registered entities:
  - applicant (join_key=SK_ID_CURR)

Registered feature views:
  - applicant_features             ( 12 features)
  - bureau_features                ( 11 features)
  - credit_history_features        ( 18 features)
  - missing_indicator_features     (  6 features)


/home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/.venv/lib/python3.14/site-packages/feast/repo_config.py:358: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(


## 5. Materialize Features to Online Store

Materialize features into the online store (SQLite) for low-latency serving.

In [7]:
from datetime import datetime, timezone

# Materialize features into the online store
# Use a time range that covers our event_timestamp (2024-01-01)
start_date = datetime(2023, 12, 31, tzinfo=timezone.utc)
end_date = datetime(2024, 1, 2, tzinfo=timezone.utc)

print(f"Materializing features from {start_date} to {end_date} ...")
fs.materialize(
    start_date=start_date,
    end_date=end_date,
)
print("\nMaterialization complete!")

# Check online store file
online_store_path = FEAST_DIR / "online_store.db"
if online_store_path.exists():
    size_mb = online_store_path.stat().st_size / 1e6
    print(f"Online store: {online_store_path} ({size_mb:.1f} MB)")

Materializing features from 2023-12-31 00:00:00+00:00 to 2024-01-02 00:00:00+00:00 ...
Materializing 4 feature views from 2023-12-31 00:00:00+00:00 to 2024-01-02 00:00:00+00:00 into the sqlite online store.

applicant_features:
bureau_features:
credit_history_features:
missing_indicator_features:

Materialization complete!
Online store: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/data/feast/online_store.db (13293.3 MB)


## 6. Retrieve Features

Demonstrate both **historical** (offline) and **online** feature retrieval.

### 6.1 Historical (Offline) Retrieval

Point-in-time correct feature retrieval for training or batch scoring.

In [8]:
# Create entity DataFrame for retrieval — sample 10 applicants
sample_ids = feast_df["SK_ID_CURR"].sample(n=10, random_state=42).tolist()

entity_df = pd.DataFrame({
    "SK_ID_CURR": sample_ids,
    "event_timestamp": pd.Timestamp("2024-01-01", tz="UTC"),
})

print("Entity DataFrame for retrieval:")
print(entity_df.to_string(index=False))

Entity DataFrame for retrieval:
 SK_ID_CURR           event_timestamp
     384575 2024-01-01 00:00:00+00:00
     214010 2024-01-01 00:00:00+00:00
     142232 2024-01-01 00:00:00+00:00
     389171 2024-01-01 00:00:00+00:00
     283617 2024-01-01 00:00:00+00:00
     362171 2024-01-01 00:00:00+00:00
     180689 2024-01-01 00:00:00+00:00
     310328 2024-01-01 00:00:00+00:00
     233043 2024-01-01 00:00:00+00:00
     232220 2024-01-01 00:00:00+00:00


In [9]:
# Retrieve historical features from offline store
feature_refs = [
    "applicant_features:AMT_INCOME_TOTAL",
    "applicant_features:AMT_CREDIT",
    "applicant_features:AMT_ANNUITY",
    "applicant_features:AGE_YEARS",
    "applicant_features:EXT_SOURCE_1",
    "applicant_features:EXT_SOURCE_2",
    "applicant_features:EXT_SOURCE_3",
    "bureau_features:BUREAU_CREDIT_COUNT",
    "bureau_features:BUREAU_AMT_CREDIT_SUM_SUM",
    "bureau_features:BUREAU_OVERDUE_CREDIT_PROPORTION",
    "credit_history_features:PREV_APPLICATION_COUNT",
    "credit_history_features:PREV_STATUS_APPROVED_COUNT",
    "credit_history_features:INST_PAYMENT_DIFF_MEAN",
]

historical_features = fs.get_historical_features(
    entity_df=entity_df,
    features=feature_refs,
).to_df()

print(f"Retrieved historical features: {historical_features.shape[0]} rows x {historical_features.shape[1]} columns")
print(f"\nColumns: {list(historical_features.columns)}")
historical_features

Retrieved historical features: 10 rows x 15 columns

Columns: ['SK_ID_CURR', 'event_timestamp', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AGE_YEARS', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'BUREAU_CREDIT_COUNT', 'BUREAU_AMT_CREDIT_SUM_SUM', 'BUREAU_OVERDUE_CREDIT_PROPORTION', 'PREV_APPLICATION_COUNT', 'PREV_STATUS_APPROVED_COUNT', 'INST_PAYMENT_DIFF_MEAN']


,SK_ID_CURR,event_timestamp,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AGE_YEARS,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,BUREAU_CREDIT_COUNT,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_OVERDUE_CREDIT_PROPORTION,PREV_APPLICATION_COUNT,PREV_STATUS_APPROVED_COUNT,INST_PAYMENT_DIFF_MEAN
0,384575,2024-01-01 00:00:00+00:00,207000.0,465457.5,52641.0,36.405201,0.675878,0.604894,0.000527,7.0,9.702765e+05,0.000000,5.0,2.0,-1151.163574
1,214010,2024-01-01 00:00:00+00:00,247500.0,1281712.5,48946.5,40.459957,0.430827,0.425351,0.712155,5.0,4.330868e+06,0.000000,12.0,10.0,-255.746567
2,142232,2024-01-01 00:00:00+00:00,202500.0,495000.0,39109.5,49.026695,0.527239,0.531760,0.207964,6.0,2.804085e+06,0.000000,7.0,3.0,-774.054504
3,389171,2024-01-01 00:00:00+00:00,247500.0,254700.0,24939.0,53.733059,NaN,0.693521,0.614414,2.0,2.525170e+05,0.000000,1.0,1.0,-864.685425
4,283617,2024-01-01 00:00:00+00:00,112500.0,308133.0,15862.5,55.652294,0.654881,0.560690,0.636376,3.0,1.360575e+06,0.000000,9.0,5.0,-3691.625977
5,362171,2024-01-01 00:00:00+00:00,85500.0,152820.0,16456.5,52.375084,NaN,0.519127,0.104795,3.0,3.696525e+05,0.333333,2.0,2.0,0.000000
6,180689,2024-01-01 00:00:00+00:00,112500.0,900000.0,24750.0,43.044491,0.714067,0.547963,NaN,2.0,1.109636e+05,0.000000,NaN,NaN,NaN
7,310328,2024-01-01 00:00:00+00:00,141606.0,810000.0,33120.0,29.571526,NaN,0.012588,0.490258,2.0,1.035000e+06,0.000000,NaN,NaN,NaN
8,233043,2024-01-01 00:00:00+00:00,130500.0,781920.0,34443.0,64.383301,NaN,0.077535,0.633032,7.0,1.455786e+06,0.000000,11.0,5.0,-958.399475
9,232220,2024-01-01 00:00:00+00:00,99000.0,900000.0,35824.5,23.969883,NaN,0.740082,NaN,NaN,NaN,NaN,11.0,5.0,-2256.402344


### 6.2 Online Retrieval

Low-latency feature lookup for real-time inference (from materialized online store).

In [10]:
# Retrieve features from the online store for a single applicant
online_features = fs.get_online_features(
    features=[
        "applicant_features:AMT_INCOME_TOTAL",
        "applicant_features:AMT_CREDIT",
        "applicant_features:AGE_YEARS",
        "applicant_features:EXT_SOURCE_2",
        "bureau_features:BUREAU_CREDIT_COUNT",
        "credit_history_features:PREV_APPLICATION_COUNT",
    ],
    entity_rows=[{"SK_ID_CURR": sample_ids[0]}],
).to_dict()

print(f"Online features for SK_ID_CURR = {sample_ids[0]}:")
print(f"{'-'*50}")
for key, values in online_features.items():
    val = values[0]
    if isinstance(val, float):
        print(f"  {key:40s} {val:>12.2f}")
    else:
        print(f"  {key:40s} {str(val):>12s}")

Online features for SK_ID_CURR = 384575:
--------------------------------------------------
  SK_ID_CURR                                     384575
  EXT_SOURCE_2                                     0.60
  AMT_INCOME_TOTAL                            207000.00
  AGE_YEARS                                       36.41
  AMT_CREDIT                                  465457.50
  BUREAU_CREDIT_COUNT                              7.00
  PREV_APPLICATION_COUNT                           5.00


### 6.3 Validate Retrieval Consistency

Verify that features retrieved from Feast match the original gold layer data.

In [11]:
# Compare Feast-retrieved features with original gold layer
check_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AGE_YEARS"]

# Get original values for the sample IDs
original = train_features[train_features["SK_ID_CURR"].isin(sample_ids)].set_index("SK_ID_CURR")
retrieved = historical_features.set_index("SK_ID_CURR")

print("Consistency check (Feast vs Gold Layer):")
print(f"{'Column':40s} {'Match':>8s}")
print(f"{'-'*50}")

all_match = True
for col in check_cols:
    orig_vals = original[col].sort_index().values
    feast_vals = retrieved[col].sort_index().values
    match = np.allclose(orig_vals, feast_vals, equal_nan=True)
    all_match = all_match and match
    print(f"  {col:38s} {'OK' if match else 'MISMATCH':>8s}")

print(f"\n{'All features match!' if all_match else 'ERRORS DETECTED — check feature definitions'}")

Consistency check (Feast vs Gold Layer):
Column                                      Match
--------------------------------------------------
  AMT_INCOME_TOTAL                             OK
  AMT_CREDIT                                   OK
  AMT_ANNUITY                                  OK
  AGE_YEARS                                    OK

All features match!


## Summary

**What we did:**
1. Loaded gold-layer engineered features from NB03 (`data/features/train_features.parquet`)
2. Prepared data for Feast by adding `event_timestamp` and `created` columns
3. Configured the Feast feature store (`feature_store.yaml` + `definitions.py`)
4. Registered 1 entity (`applicant`) and 4 feature views:
   - `applicant_features` — core application-level features
   - `bureau_features` — credit bureau aggregation features
   - `credit_history_features` — previous app, POS, credit card, installment features
   - `missing_indicator_features` — 6 curated binary `_IS_MISSING` flags for features with meaningful missingness (>5%)
5. Materialized features to the online store (SQLite) for low-latency serving
6. Demonstrated historical and online feature retrieval
7. Validated that retrieved features match the original gold layer data

**Feature store artifacts:**
- Registry: `data/feast/registry.db`
- Online store: `data/feast/online_store.db`
- Feast-ready features: `data/feast/train_features.parquet`

**Next:** NB05 — Baseline Models (LogReg, Random Forest) + MLFlow experiment tracking

## 8. Ad-Hoc Feature Explorer

Use the cells below to interactively query the feature store — look up any applicant by `SK_ID_CURR` and retrieve any registered feature.

In [12]:
# ── Ad-hoc feature query ──────────────────────────────────────────────
# Change these two variables and re-run the cell to explore any applicant/features.

QUERY_IDS = [100002, 100003, 100004]  # <-- SK_ID_CURR(s) to look up

QUERY_FEATURES = [                     # <-- feature_view:feature pairs
    "applicant_features:AMT_INCOME_TOTAL",
    "applicant_features:AMT_CREDIT",
    "applicant_features:AGE_YEARS",
    "applicant_features:EXT_SOURCE_1",
    "applicant_features:EXT_SOURCE_2",
    "applicant_features:EXT_SOURCE_3",
    "bureau_features:BUREAU_CREDIT_COUNT",
    "bureau_features:BUREAU_AMT_CREDIT_SUM_SUM",
    "bureau_features:BUREAU_OVERDUE_CREDIT_PROPORTION",
    "credit_history_features:PREV_APPLICATION_COUNT",
    "credit_history_features:PREV_STATUS_APPROVED_COUNT",
    "credit_history_features:INST_PAYMENT_DIFF_MEAN",
    # Missing indicators (top 6)
    "missing_indicator_features:EXT_SOURCE_1_IS_MISSING",
    "missing_indicator_features:EXT_SOURCE_3_IS_MISSING",
    "missing_indicator_features:OWN_CAR_AGE_IS_MISSING",
    "missing_indicator_features:OCCUPATION_TYPE_IS_MISSING",
    "missing_indicator_features:BUREAU_CREDIT_COUNT_IS_MISSING",
    "missing_indicator_features:PREV_APPLICATION_COUNT_IS_MISSING",
]

# ── Online store lookup (low latency) ────────────────────────────────
entity_rows = [{"SK_ID_CURR": sid} for sid in QUERY_IDS]
result = fs.get_online_features(features=QUERY_FEATURES, entity_rows=entity_rows)
result_df = pd.DataFrame(result.to_dict()).set_index("SK_ID_CURR")

print(f"Online store query — {len(QUERY_IDS)} applicant(s), {len(QUERY_FEATURES)} features\n")
result_df

Online store query — 3 applicant(s), 18 features



,AGE_YEARS,EXT_SOURCE_3,EXT_SOURCE_2,AMT_CREDIT,EXT_SOURCE_1,AMT_INCOME_TOTAL,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_OVERDUE_CREDIT_PROPORTION,BUREAU_CREDIT_COUNT,INST_PAYMENT_DIFF_MEAN,PREV_APPLICATION_COUNT,PREV_STATUS_APPROVED_COUNT,OCCUPATION_TYPE_IS_MISSING,EXT_SOURCE_3_IS_MISSING,EXT_SOURCE_1_IS_MISSING,OWN_CAR_AGE_IS_MISSING,PREV_APPLICATION_COUNT_IS_MISSING,BUREAU_CREDIT_COUNT_IS_MISSING
SK_ID_CURR,,,,,,,,,,,,,,,,,,
100002,25.902807,0.139376,0.262949,406597.5,0.083037,202500.0,8.650556e+05,0.0,8.0,0.0,1.0,1.0,0,0,0,1,0,0
100003,45.900070,NaN,0.622246,1293502.5,0.311267,270000.0,1.017400e+06,0.0,4.0,0.0,3.0,3.0,0,1,0,1,0,0
100004,52.145107,0.729567,0.555912,135000.0,NaN,67500.0,1.890378e+05,0.0,2.0,0.0,1.0,1.0,0,0,1,0,0,0
